# Full harmonization v0.5 sobre los 47 PS+TT del audit v2.3

Generaliza el piloto v0.4 (CMAPSS, `tail_policy='pad'`, asserts duros) a los 47 datasets relevantes del audit v2.3 (36 PRETRAIN_SOURCE + 11 TRANSFER_TARGET). Cada dataset se procesa con la misma pipeline channel-independent del piloto y produce los mismos artefactos por dataset (`shards`, `manifest.json`, `anti_leakage_report.json`, `warnings_detail.jsonl`, `done.flag`) en Drive, mas filas agregadas en `results/processed_summary*.csv` y `results/client_summary*.csv`.

Modo de ejecucion (`RUN_MODE`):

- `dry_run`: **no carga PHMD, no escribe shards**. Genera el plan de ejecucion (47 filas) en `results/harmonization_full_plan.csv` con tamanos estimados, accion prevista (PROCESS / SKIP / FORCE / STALE) y motivo. Para verificar configuracion antes del full.
- `smoke`: procesa solo `SMOKE_DATASETS` (5 datasets pequenos por defecto). **Escribe en rutas separadas** (`processed/_smoke_v0_5/<DATASET>/`, `results/processed_summary_smoke.csv`, `results/client_summary_smoke.csv`). No toca artefactos canonicos. Pensado para validar el codigo en datos reales antes del full.
- `full`: procesa los 47 PS+TT y escribe en rutas canonicas. Requiere que el smoke haya pasado.

Reentrancia segura (orden corregido v0.5):

1. cargar dataset via phmd
2. detectar columnas (signal, target, subset_id)
3. construir `pipeline_config` y calcular `pipeline_config_hash`
4. validar manifest existente: hash + dataset/audit/W/S/P/mask coinciden, done.flag existe, shards declarados existen, metricas clave coinciden con audit
5. **solo entonces** decidir SKIP_VALID o procesar
6. procesar siempre en `_staging_v0_5/<DATASET>__<run_id>/`
7. promover staging a la ruta final (canonical o smoke) **solo si todos los asserts pasan**

Sin borrado destructivo de carpetas validas hasta que la nueva version pase. Sin acumular samples en memoria: streaming via `ShardWriter`, contadores incrementales, buffer minimo para validar contrato y mask ratio. `done.flag` es el ultimo artefacto escrito; el manifest NO contiene `done=True`.

Politica de output:

- shards `.tar` van a Drive, NO se versionan en git.
- `processed_summary*.csv`, `client_summary*.csv` y `harmonization_full_plan.csv` van a `results/`, ligeros y versionables.
- Este notebook NO ejecuta `git commit` ni `git push`.


## Setup

Montamos Drive y lanzamos `colab_init.sh`. Si la salida no termina con `Setup OK`, ejecutar primero `setup/colab_bootstrap.ipynb`.


In [ ]:
from google.colab import drive; drive.mount('/content/drive')
!bash /content/drive/MyDrive/fm_fl_phmd/colab_init.sh


## 1. Configuracion del full v0.5

Constantes globales heredadas del audit v2.3. `RUN_MODE` controla todo el comportamiento:

- `dry_run`: solo plan, ni carga datos ni escribe shards.
- `smoke`: solo `SMOKE_DATASETS`, escritura en `_smoke_v0_5/` y summaries con sufijo `_smoke`.
- `full`: los 47 PS+TT, escritura canonica.

`SMOKE_DATASETS` recomendado cubre: TT primary con padding alto (CMAPSS), PS pequeno con client assignment (DUS20), TT muy pequeno (CBM14), TT RUL pequeno (PHME20), TT clasificacion 1 canal (PBCP16). `EDGE_DATASETS` no se procesan automaticamente; estan listados como referencia para casos limite a inspeccionar manualmente (padding moderado, datasets dense-vs-valid extremos).


In [ ]:
from pathlib import Path

# ---------------------------------------------------------------------------
# Modo de ejecucion
# ---------------------------------------------------------------------------
RUN_MODE = "dry_run"   # "dry_run" | "smoke" | "full"

# Subconjunto del smoke: TT primary (CMAPSS) + PS pequeno (DUS20) + TT pequenos
SMOKE_DATASETS = ["CMAPSS", "DUS20", "CBM14", "PHME20", "PBCP16"]
# Casos limite a inspeccionar manualmente tras el smoke (no se procesan auto)
EDGE_DATASETS  = ["PHM14", "PHM18", "PHMAP23"]

# Reentrancia
FORCE_REPROCESS_ALL      = False
FORCE_REPROCESS_DATASETS = set()   # subconjunto a forzar; vacio = decide hash full

# Politica de fallos
FAIL_FAST = True                   # smoke: True. full: opcional False para diagnosticar.

# Limites operativos
MAX_DATASETS = None                # truncar el orden (p.ej. 5 para test rapido); None = todos

# Reproducibilidad
SEED = 42

# ---------------------------------------------------------------------------
# Identidad del run
# ---------------------------------------------------------------------------
PIPELINE_VERSION = "0.5"

# Heredadas del audit v2.3 (no se cambian)
AUDIT_VERSION_ESPERADO = "2.3"
TAIL_POLICY            = "pad"

# Hiperparametros (CLAUDE.md sec.12)
WINDOW_SIZE    = 512
STRIDE         = 256
PATCH_SIZE     = 16
N_PATCHES      = WINDOW_SIZE // PATCH_SIZE
MASK_RATIO     = 0.3
SHARD_MAXCOUNT = 5000

# Limpieza NaN
MAX_NAN_PCT_UNIDAD = 30.0

# Overrides (mismos que audit y piloto)
SUBSET_ID_OVERRIDE  = {"CMAPSS": "FD"}
TARGET_COL_OVERRIDE = {"UNIBO21": "soc"}

# Schema del manifest
MANIFEST_SCHEMA_VERSION = "1.0"

# ---------------------------------------------------------------------------
# Rutas en Drive y repo
# ---------------------------------------------------------------------------
DRIVE_ROOT     = Path('/content/drive/MyDrive/fm_fl_phmd')
RAW_DIR        = DRIVE_ROOT / 'raw'
PROCESSED_ROOT_CANONICAL = DRIVE_ROOT / 'processed'
PROCESSED_ROOT_SMOKE     = DRIVE_ROOT / 'processed' / '_smoke_v0_5'
STAGING_ROOT             = DRIVE_ROOT / 'processed' / '_staging_v0_5'

REPO              = Path('/content/fm_fl_phmd')
AUDIT_CSV         = REPO / 'results' / 'audit' / 'audit_report.csv'
AUDIT_SUMMARY     = REPO / 'results' / 'audit' / 'audit_summary.json'
AUDIT_GROUPS      = REPO / 'results' / 'audit' / 'audit_groups.json'

# Outputs canonical / smoke / plan
PLAN_CSV                  = REPO / 'results' / 'harmonization_full_plan.csv'
PROCESSED_SUMMARY_CANON   = REPO / 'results' / 'processed_summary.csv'
CLIENT_SUMMARY_CANON      = REPO / 'results' / 'client_summary.csv'
PROCESSED_SUMMARY_SMOKE   = REPO / 'results' / 'processed_summary_smoke.csv'
CLIENT_SUMMARY_SMOKE      = REPO / 'results' / 'client_summary_smoke.csv'

# Selectores derivados (se confirman tras preflight)
WRITE_CANONICAL_OUTPUTS = (RUN_MODE == "full")

PHMD_HOME_CACHE = '/root/.phmd'

# Logs
LOG_DIR = DRIVE_ROOT / 'logs' / 'harmonization_full'
LOG_DIR.mkdir(parents=True, exist_ok=True)

# run_id (se completa en cell del bucle)
import datetime as _dt
RUN_ID = _dt.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

print(f'RUN_MODE                : {RUN_MODE}')
print(f'PIPELINE_VERSION        : {PIPELINE_VERSION}')
print(f'AUDIT_VERSION_ESPERADO  : {AUDIT_VERSION_ESPERADO}')
print(f'TAIL_POLICY             : {TAIL_POLICY}')
print(f'W={WINDOW_SIZE} S={STRIDE} P={PATCH_SIZE} N={N_PATCHES} mask={MASK_RATIO}')
print(f'WRITE_CANONICAL_OUTPUTS : {WRITE_CANONICAL_OUTPUTS}')
print(f'FAIL_FAST               : {FAIL_FAST}')
print(f'SEED                    : {SEED}')
print(f'RUN_ID                  : {RUN_ID}')
if RUN_MODE == "smoke":
    print(f'SMOKE_DATASETS          : {SMOKE_DATASETS}')
print(f'FORCE_REPROCESS_ALL      : {FORCE_REPROCESS_ALL}')
print(f'FORCE_REPROCESS_DATASETS : {sorted(FORCE_REPROCESS_DATASETS) if FORCE_REPROCESS_DATASETS else "(vacio)"}')


## 2. Preflight contra audit v2.3 (asserts duros)

Validamos `audit_report.csv`, `audit_summary.json` y `audit_groups.json` simultaneamente. Si algo no cuadra, abortamos antes de tocar nada.

Lecturas obligatorias y asserts:

- `audit_summary["audit_version"] == "2.3"`
- `audit_summary["tail_policy"] == "pad"`
- `audit_summary["roles"] == {"DROP":4, "EXCLUDED":2, "PRETRAIN_SOURCE":36, "TRANSFER_TARGET":11}`
- df_audit filtrado a PS+TT tiene 47 filas (36 PS + 11 TT)
- todos los 47 tienen `tail_policy == "pad"` y `audit_version == "2.3"`
- todas las metricas clave del audit estan no-nulas en los 47
- `len(audit_groups["clients"]) == 10` y todos los datasets PS estan asignados a un cliente

Se construye `dataset_to_client` para el `client_summary`. TRANSFER_TARGET no entra en clientes de pretraining.


In [ ]:
import json
import pandas as pd

assert AUDIT_CSV.exists(),     f'No existe {AUDIT_CSV}'
assert AUDIT_SUMMARY.exists(), f'No existe {AUDIT_SUMMARY}'
assert AUDIT_GROUPS.exists(),  f'No existe {AUDIT_GROUPS}'

# audit_summary.json
audit_summary = json.loads(AUDIT_SUMMARY.read_text())
assert audit_summary.get("audit_version") == AUDIT_VERSION_ESPERADO, (
    f'audit_summary.audit_version={audit_summary.get("audit_version")} != {AUDIT_VERSION_ESPERADO}'
)
assert audit_summary.get("tail_policy") == TAIL_POLICY, (
    f'audit_summary.tail_policy={audit_summary.get("tail_policy")} != {TAIL_POLICY}'
)
expected_roles = {"DROP": 4, "EXCLUDED": 2, "PRETRAIN_SOURCE": 36, "TRANSFER_TARGET": 11}
assert audit_summary.get("roles") == expected_roles, (
    f'audit_summary.roles={audit_summary.get("roles")} != {expected_roles}'
)
print(f'audit_summary: v={audit_summary["audit_version"]}, '
      f'tail_policy={audit_summary["tail_policy"]}, roles={audit_summary["roles"]}')

# Totales PS para validacion global del full
AUDIT_TOTALS_PS = {
    'total_temporal_patches_ps':       audit_summary.get('total_temporal_patches_ps'),
    'total_channel_patches_ps':        audit_summary.get('total_channel_patches_ps'),
    'total_dense_temporal_patches_ps': audit_summary.get('total_dense_temporal_patches_ps'),
    'total_dense_channel_patches_ps':  audit_summary.get('total_dense_channel_patches_ps'),
}

# audit_groups.json
audit_groups = json.loads(AUDIT_GROUPS.read_text())
assert audit_groups.get("audit_version") == AUDIT_VERSION_ESPERADO
assert audit_groups.get("tail_policy")   == TAIL_POLICY
clients_dict = audit_groups.get("clients", {})
assert len(clients_dict) == 10, f'audit_groups.clients tiene {len(clients_dict)} clientes, esperaba 10'
print(f'audit_groups: {len(clients_dict)} clientes')

# Mapping dataset -> cliente (solo PS estan presentes en audit_groups)
dataset_to_client = {}
for cliente, info in clients_dict.items():
    for d in info.get('datasets', []):
        dataset_to_client[d['dataset']] = cliente

# audit_report.csv y filtrado de PS+TT
df_audit = pd.read_csv(AUDIT_CSV)
df_relevant = df_audit[df_audit['role'].isin(['PRETRAIN_SOURCE', 'TRANSFER_TARGET'])].copy()
n_ps = int((df_relevant['role'] == 'PRETRAIN_SOURCE').sum())
n_tt = int((df_relevant['role'] == 'TRANSFER_TARGET').sum())
assert n_ps == 36, f'n_ps={n_ps} != 36'
assert n_tt == 11, f'n_tt={n_tt} != 11'
assert len(df_relevant) == 47, f'df_relevant={len(df_relevant)} != 47'
print(f'audit_report: 47 PS+TT ({n_ps} PS + {n_tt} TT)')

# Cada PS+TT debe tener tail_policy y audit_version coherentes
versions_ps_tt = df_relevant['audit_version'].astype(str).unique().tolist()
assert versions_ps_tt == [AUDIT_VERSION_ESPERADO], f'audit_version PS+TT: {versions_ps_tt}'
tails_ps_tt = df_relevant['tail_policy'].astype(str).unique().tolist()
assert tails_ps_tt == [TAIL_POLICY], f'tail_policy PS+TT: {tails_ps_tt}'

# Metricas audit obligatorias no-nulas en los 47
cols_obligatorias = [
    'n_filas_total', 'n_canales', 'n_windows',
    'n_temporal_patches', 'n_channel_patches',
    'n_dense_temporal_patches', 'n_dense_channel_patches',
    'padding_ratio', 'invalid_patch_ratio', 'dense_vs_valid_ratio',
    'estimated_n_shards',
]
for col in cols_obligatorias:
    n_null = df_relevant[col].isna().sum()
    assert n_null == 0, (
        f'audit_report.csv: columna {col} tiene {n_null} valores nulos en PS+TT. '
        f'Datasets afectados: {df_relevant.loc[df_relevant[col].isna(), "dataset"].tolist()}'
    )
print(f'audit_report: las {len(cols_obligatorias)} metricas obligatorias estan no-nulas en los 47')

# Cada PS tiene un cliente asignado
ps_sin_cliente = [d for d in df_relevant.loc[df_relevant['role']=='PRETRAIN_SOURCE','dataset']
                  if d not in dataset_to_client]
assert not ps_sin_cliente, f'PRETRAIN_SOURCE sin cliente asignado: {ps_sin_cliente}'
print(f'dataset_to_client: {len(dataset_to_client)} PS mapeados a {len(clients_dict)} clientes')

# Selector segun RUN_MODE
if RUN_MODE == "smoke":
    datasets_a_procesar = [d for d in SMOKE_DATASETS if d in set(df_relevant['dataset'])]
    faltantes = [d for d in SMOKE_DATASETS if d not in set(df_relevant['dataset'])]
    if faltantes:
        print(f'AVISO: SMOKE_DATASETS no en PS+TT: {faltantes}')
elif RUN_MODE == "full":
    datasets_a_procesar = sorted(df_relevant['dataset'].tolist())
elif RUN_MODE == "dry_run":
    datasets_a_procesar = sorted(df_relevant['dataset'].tolist())
else:
    raise ValueError(f'RUN_MODE invalido: {RUN_MODE}')

if MAX_DATASETS is not None and MAX_DATASETS > 0:
    datasets_a_procesar = datasets_a_procesar[:MAX_DATASETS]
    print(f'MAX_DATASETS={MAX_DATASETS}: truncamos a {len(datasets_a_procesar)} datasets.')

print()
print(f'Datasets a procesar en modo {RUN_MODE}: {len(datasets_a_procesar)}')
print(f'Lista: {datasets_a_procesar[:20]}{"..." if len(datasets_a_procesar) > 20 else ""}')


## 3. Loader phmd (mismo patron v2.2.2 del audit)

Copia `.zip` de Drive a `/root/.phmd/datasets/`, inyecta `zipfile` en submodulos `phmd` antes de cada carga (workaround PRONOSTIA), y limpia los ficheros del dataset al terminar. NO limpia `/root/.phmd` entero entre datasets.

En `RUN_MODE="dry_run"` no se llama a esta carga.


In [ ]:
import shutil
import sys
import zipfile
from phmd import datasets as phmd_datasets

OVERRIDE_FILE_PREFIX = {
    'JNUB':       'JNUBearing',
    'MFPT':       'MPFT',
    'PHM19':      'PHM2019',
    'SSPSNASA15': 'SSPSNASA2015',
}


def inyectar_zipfile_en_phmd():
    for nombre_mod, mod in list(sys.modules.items()):
        if nombre_mod.startswith('phmd') and mod is not None:
            try:
                setattr(mod, 'zipfile', zipfile)
            except Exception:
                pass


inyectar_zipfile_en_phmd()


def archivos_dataset(nombre):
    drive_dir = RAW_DIR / 'datasets'
    if not drive_dir.exists():
        return []
    objetivo = OVERRIDE_FILE_PREFIX.get(nombre, nombre).lower()
    encontrados = []
    for p in drive_dir.iterdir():
        if not p.is_file():
            continue
        stem = p.stem.lower()
        name = p.name.lower()
        if (stem == objetivo
            or stem.startswith(objetivo + '_')
            or name.startswith(objetivo + '.')):
            encontrados.append(p); continue
        if stem.startswith(objetivo) and len(stem) > len(objetivo):
            siguiente = stem[len(objetivo)]
            if siguiente.isalpha():
                encontrados.append(p)
    return encontrados


def cargar_dataset(nombre, verbose=False):
    """Copia .zip(s) a /root/.phmd/datasets/ y carga via phmd."""
    inyectar_zipfile_en_phmd()
    phmd_cache = Path(PHMD_HOME_CACHE) / 'datasets'
    phmd_cache.mkdir(parents=True, exist_ok=True)
    ficheros_drive = archivos_dataset(nombre)
    if not ficheros_drive:
        raise FileNotFoundError(f'No se encuentran ficheros del dataset {nombre} en Drive')
    ficheros_copiados = []
    for f in ficheros_drive:
        destino = phmd_cache / f.name
        try:
            tamano_origen = f.stat().st_size
        except Exception:
            tamano_origen = -1
        recopiar = (not destino.exists()) or (
            tamano_origen > 0 and destino.stat().st_size != tamano_origen
        )
        if recopiar:
            shutil.copy(f, destino)
        if destino.stat().st_size == 0:
            raise IOError(f'Copia incompleta a {destino} (0 bytes)')
        if verbose:
            print(f'  [copiado] {f.name} ({destino.stat().st_size} bytes)')
        ficheros_copiados.append(destino)
    try:
        ds = phmd_datasets.Dataset(nombre, cache_dir=PHMD_HOME_CACHE)
    except NameError as ne:
        if 'zipfile' in str(ne).lower():
            inyectar_zipfile_en_phmd()
            ds = phmd_datasets.Dataset(nombre, cache_dir=PHMD_HOME_CACHE)
        else:
            raise
    if not getattr(ds, 'tasks', None):
        raise RuntimeError(f'phmd no expone tareas para {nombre}')
    fold = ds.tasks[0][0]
    if isinstance(fold, dict):
        splits = {k: v for k, v in fold.items() if v is not None}
    elif isinstance(fold, (tuple, list)):
        nombres_split = ['train', 'val', 'test'][:len(fold)]
        splits  = {n: v for n, v in zip(nombres_split, fold) if v is not None}
    else:
        splits = {'train': fold}
    return ds, splits, ficheros_copiados


def limpiar_cache_dataset(nombre, ficheros_copiados):
    for f in (ficheros_copiados or []):
        try:
            if f.exists() and f.is_file():
                f.unlink()
        except Exception:
            pass
    phmd_datasets_dir = Path(PHMD_HOME_CACHE) / 'datasets'
    nombres_posibles = {nombre, OVERRIDE_FILE_PREFIX.get(nombre, nombre)}
    for nombre_posible in nombres_posibles:
        candidato = phmd_datasets_dir / nombre_posible
        if candidato.exists() and candidato.is_dir():
            shutil.rmtree(candidato, ignore_errors=True)


print('Loader v0.5 definido (mismo patron del audit v2.2.2).')


## 4. Helpers de identidad robustos

`safe_slug` acepta strings, ints, floats enteros y `NaN`. Construye IDs deterministas sin asumir tipos. `to_jsonable` normaliza valores numpy/pandas a tipos serializables JSON sin NaN.

Patron de IDs:

```
trajectory_id     = "{dataset}__subset-{slug(subset)}__split-{split}__unit-{slug(unit)}"
unit_global_id    = trajectory_id   (alias)
asset_key         = "{dataset}__subset-{slug(subset)}__unit-{slug(unit)}"   # sin split
```


In [ ]:
import math
import numpy as np


def safe_slug(value):
    """Convierte cualquier valor de unit_id/subset_id a slug determinista.

    - None / NaN -> 'none'
    - int / np.integer -> str(int(...))
    - float entero -> str(int(...))
    - float no entero -> reemplaza '.' y '-' por marcadores
    - str -> caracteres no alfanumericos -> '_'
    """
    if value is None:
        return 'none'
    if isinstance(value, (int, np.integer)) and not isinstance(value, bool):
        return str(int(value))
    if isinstance(value, (float, np.floating)):
        fv = float(value)
        if math.isnan(fv):
            return 'none'
        if fv.is_integer():
            return str(int(fv))
        return str(fv).replace('.', 'p').replace('-', 'n')
    if isinstance(value, np.bool_):
        return 'true' if bool(value) else 'false'
    s = str(value)
    return ''.join(c if c.isalnum() else '_' for c in s) or 'none'


def to_jsonable(obj):
    """Convierte numpy/pandas types a tipos JSON-serializables sin NaN."""
    if obj is None:
        return None
    if isinstance(obj, bool):
        return obj
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    if isinstance(obj, (int, np.integer)):
        return int(obj)
    if isinstance(obj, (float, np.floating)):
        v = float(obj)
        if not math.isfinite(v):
            return None
        return v
    if isinstance(obj, str):
        return obj
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, np.ndarray):
        return [to_jsonable(x) for x in obj.tolist()]
    if isinstance(obj, dict):
        return {str(k): to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [to_jsonable(v) for v in obj]
    try:
        return str(obj)
    except Exception:
        return None


def _trajectory_id(dataset, split_name, subset_val, unit_val):
    return f'{dataset}__subset-{safe_slug(subset_val)}__split-{split_name}__unit-{safe_slug(unit_val)}'


def _asset_key(dataset, subset_val, unit_val):
    return f'{dataset}__subset-{safe_slug(subset_val)}__unit-{safe_slug(unit_val)}'


print('Helpers safe_slug, to_jsonable y constructores de IDs definidos.')


## 5. Deteccion de columnas y construccion de trajectory_id

Mismo detector del audit/piloto, con overrides `SUBSET_ID_OVERRIDE` y `TARGET_COL_OVERRIDE`. Tras detectar columnas, validamos contra el audit (target_col y subset_id_col esperados) y construimos `trajectory_id`, `unit_global_id` (alias) y `asset_key_without_split` en cada split.


In [ ]:
import pandas as pd


def detectar_columnas(ds, splits, nombre):
    meta = getattr(ds, 'metadata', None) or getattr(ds, 'meta', None) or {}
    if not isinstance(meta, dict):
        meta = {}
    df = None
    for k in ('train', 'val', 'test'):
        if k in splits and splits[k] is not None and len(splits[k]) > 0:
            df = splits[k]; break
    if df is None:
        return {'unit_col': None, 'subset_id_col': None,
                'target_col_seleccionado': None, 'target_candidates': [],
                'time_col': None, 'signal_cols': [], 'metadata_cols': []}
    cols = list(df.columns)

    unit_col = meta.get('unit_id') or meta.get('group_col') or meta.get('unit')
    if unit_col not in cols:
        unit_col = None
    if unit_col is None:
        for c in ('unit', 'unit_id', 'id', 'engine_id', 'machine_id',
                  'experiment', 'series_id', 'unit_number'):
            if c in cols:
                unit_col = c; break
        else:
            for c in cols:
                if df[c].dtype.kind in 'iuO' and df[c].nunique() < len(df) / 2:
                    unit_col = c; break

    time_col = meta.get('time_col') or meta.get('cycle') or meta.get('time')
    if time_col not in cols:
        time_col = None
    if time_col is None:
        for c in ('time', 'cycle', 'timestamp', 'time_in_cycles', 't', 'step'):
            if c in cols:
                time_col = c; break

    subset_id_col = SUBSET_ID_OVERRIDE.get(nombre)
    if subset_id_col is not None and subset_id_col not in cols:
        subset_id_col = None

    candidates = []
    try:
        for t in (ds.tasks or []):
            nt = getattr(t, 'name', None) or getattr(t, 'task_name', None)
            if nt and str(nt) in cols and str(nt) not in candidates:
                candidates.append(str(nt))
    except Exception:
        pass
    # OJO: lista exactamente alineada con audit v2.3. No anadir 'soc' aqui:
    # en datasets de bateria con target='rul' (CALCE_CS2, CALCE_CX2, FCLB19, NB1, NB14)
    # 'soc' es una senal fisica legitima, no un target candidate. UNIBO21 lo trata
    # como target via TARGET_COL_OVERRIDE, que se inyecta despues de esta lista.
    for c in ('rul', 'RUL', 'target', 'label', 'class', 'fault',
              'state', 'condition', 'anomaly', 'y', 'wear'):
        if c in cols and c not in candidates:
            candidates.append(c)

    target_override = TARGET_COL_OVERRIDE.get(nombre)
    if target_override and target_override in cols:
        if target_override in candidates:
            candidates.remove(target_override)
        candidates.insert(0, target_override)
        target_col_seleccionado = target_override
    else:
        target_col_seleccionado = candidates[0] if candidates else None

    descartar = {c for c in (unit_col, time_col, target_col_seleccionado, subset_id_col)
                 if c is not None}
    descartar |= set(candidates)
    signal_cols = [c for c in cols
                   if c not in descartar and df[c].dtype.kind in 'fiub']

    metadata_cols = [c for c in (unit_col, subset_id_col) if c is not None]

    return {
        'unit_col':                unit_col,
        'subset_id_col':           subset_id_col,
        'target_col_seleccionado': target_col_seleccionado,
        'target_candidates':       candidates,
        'time_col':                time_col,
        'signal_cols':             signal_cols,
        'metadata_cols':           metadata_cols,
    }


def validar_columnas_contra_audit(columnas, fila_audit, dataset):
    """Compara target_col y subset_id_col detectados contra los del audit.
    Falla si difieren salvo override documentado."""
    audit_target  = fila_audit.get('target_col')
    audit_subset  = fila_audit.get('subset_id_col_detected')
    audit_canales = int(fila_audit['n_canales']) if pd.notna(fila_audit.get('n_canales')) else None

    if pd.notna(audit_target) and columnas['target_col_seleccionado'] != audit_target:
        raise RuntimeError(
            f'{dataset}: target_col detectado={columnas["target_col_seleccionado"]!r} != '
            f'audit={audit_target!r}'
        )
    if pd.notna(audit_subset) and audit_subset and columnas.get('subset_id_col') != audit_subset:
        raise RuntimeError(
            f'{dataset}: subset_id_col detectado={columnas.get("subset_id_col")!r} != '
            f'audit={audit_subset!r}'
        )
    # Asserts duros
    target_col    = columnas['target_col_seleccionado']
    subset_id_col = columnas.get('subset_id_col')
    time_col      = columnas.get('time_col')
    unit_col      = columnas['unit_col']
    signal_cols   = columnas['signal_cols']
    if target_col is not None:
        assert target_col not in signal_cols, f'{dataset}: target_col en signal_cols'
    if subset_id_col is not None:
        assert subset_id_col not in signal_cols, f'{dataset}: subset_id_col en signal_cols'
    if time_col is not None:
        assert time_col not in signal_cols, f'{dataset}: time_col en signal_cols'
    if unit_col is not None:
        assert unit_col not in signal_cols, f'{dataset}: unit_col en signal_cols'
    if audit_canales is not None:
        assert len(signal_cols) == audit_canales, (
            f'{dataset}: n_signal_cols={len(signal_cols)} != audit_n_canales={audit_canales}'
        )


def construir_trajectory_id(splits, columnas, dataset):
    unit_col      = columnas['unit_col']
    subset_id_col = columnas.get('subset_id_col')
    if unit_col is None:
        # Fallback alineado con audit v2.3 (cell metricas_forma_volumen):
        # cuando el detector heuristico no encuentra unit_col, el audit asume
        # 1 trayectoria por split (n_unidades_total = n_splits). Replicamos
        # ese comportamiento creando una unidad sintetica constante por split.
        # PHM15 es el caso conocido (3 splits = 3 trayectorias en audit).
        unit_col = '__synthetic_unit__'
        columnas['unit_col'] = unit_col
        for split_name in list(splits.keys()):
            df_sin = splits[split_name].copy()
            df_sin[unit_col] = 'all'
            splits[split_name] = df_sin
    for split_name in list(splits.keys()):
        df = splits[split_name].copy()
        if subset_id_col and subset_id_col in df.columns:
            df['trajectory_id'] = df.apply(
                lambda r: _trajectory_id(dataset, split_name, r[subset_id_col], r[unit_col]),
                axis=1)
            df['asset_key_without_split'] = df.apply(
                lambda r: _asset_key(dataset, r[subset_id_col], r[unit_col]),
                axis=1)
        else:
            df['trajectory_id'] = df.apply(
                lambda r: _trajectory_id(dataset, split_name, None, r[unit_col]),
                axis=1)
            df['asset_key_without_split'] = df.apply(
                lambda r: _asset_key(dataset, None, r[unit_col]),
                axis=1)
        # alias retro-compat
        df['unit_global_id'] = df['trajectory_id']
        splits[split_name] = df
    return splits


print('Detector + validacion contra audit + trajectory_id definidos.')


## 6. Helpers de pipeline (val_fallback, NaN, ordenamiento, ventaneo, normalizacion, patches)

Mismas funciones del piloto v0.4. `ventanas_de_unidad` con `tail_policy='pad'`. `normalizar_instancia` ignora padding y persiste `std_used`. `patches_C_N_P` produce `(C, N, P)` + `valid_patch_mask`.


In [ ]:
import numpy as np


def aplicar_val_fallback_si_falta(splits, columnas, dataset, ratio=0.10, seed=SEED):
    if 'val' in splits and splits['val'] is not None and len(splits['val']) > 0:
        return splits, {'aplicado': False, 'razon': 'phmd entrega val directamente'}
    if 'train' not in splits or splits['train'] is None or len(splits['train']) == 0:
        return splits, {'aplicado': False, 'razon': 'no hay train'}
    df_train = splits['train']
    trajs_train = df_train['trajectory_id'].unique()
    rng = np.random.default_rng(seed)
    n_val = max(1, int(round(ratio * len(trajs_train))))
    trajs_val = set(rng.choice(trajs_train, size=n_val, replace=False))
    df_val_new   = df_train[df_train['trajectory_id'].isin(trajs_val)].copy()
    df_train_new = df_train[~df_train['trajectory_id'].isin(trajs_val)].copy()
    def _reasignar(df, split_destino):
        df = df.copy()
        unit_col      = columnas['unit_col']
        subset_id_col = columnas.get('subset_id_col')
        if subset_id_col and subset_id_col in df.columns:
            df['trajectory_id'] = df.apply(
                lambda r: _trajectory_id(dataset, split_destino, r[subset_id_col], r[unit_col]),
                axis=1)
        else:
            df['trajectory_id'] = df.apply(
                lambda r: _trajectory_id(dataset, split_destino, None, r[unit_col]),
                axis=1)
        df['unit_global_id'] = df['trajectory_id']
        return df
    df_val_new   = _reasignar(df_val_new,   'val')
    df_train_new = _reasignar(df_train_new, 'train')
    splits['val']   = df_val_new
    splits['train'] = df_train_new
    return splits, {
        'aplicado':                True,
        'criterio':                'trajectory_id (no unit_col crudo)',
        'n_trayectorias_movidas':  len(trajs_val),
        'n_trayectorias_train_restante': df_train_new['trajectory_id'].nunique(),
        'seed':                    seed,
        'ratio_objetivo':          ratio,
    }


def limpiar_nans_y_ordenar(splits, columnas):
    unit_col    = columnas['unit_col']
    signal_cols = columnas['signal_cols']
    time_col    = columnas.get('time_col')

    for split_name in list(splits.keys()):
        df = splits[split_name].copy()
        df['_source_row_order'] = df.groupby('trajectory_id', sort=False).cumcount()
        splits[split_name] = df

    source_rows_por_split = {sn: int(len(splits[sn])) for sn in splits.keys()}
    source_rows_total     = int(sum(source_rows_por_split.values()))

    unidades_descartadas_total = 0
    for split_name in list(splits.keys()):
        df_split = splits[split_name]
        dfs_unidades = []
        n_desc = 0
        for uid, sub in df_split.groupby('trajectory_id', sort=False):
            X = sub[signal_cols].to_numpy() if signal_cols else np.empty((len(sub), 0))
            if X.size > 0 and np.isnan(X).any():
                nan_pct = np.isnan(X).mean(axis=0) * 100
                if (nan_pct > MAX_NAN_PCT_UNIDAD).any():
                    n_desc += 1
                    continue
                sub = sub.copy()
                sub[signal_cols] = sub[signal_cols].interpolate(
                    method='linear', axis=0, limit_direction='both')
            dfs_unidades.append(sub)
        out = pd.concat(dfs_unidades, ignore_index=True) if dfs_unidades else df_split.iloc[0:0]
        splits[split_name] = out
        unidades_descartadas_total += n_desc

    if time_col is not None and all(time_col in splits[sn].columns for sn in splits if splits[sn] is not None):
        order_cols = ['trajectory_id', time_col]
        ordered_by_unit_and_cycle = True
        ordered_by_unit_and_source_order = False
    else:
        order_cols = ['trajectory_id', '_source_row_order']
        ordered_by_unit_and_cycle = False
        ordered_by_unit_and_source_order = True

    for split_name in list(splits.keys()):
        splits[split_name] = (
            splits[split_name]
            .sort_values(order_cols, kind='mergesort')
            .reset_index(drop=True)
        )

    return splits, {
        'source_rows_total':                source_rows_total,
        'source_rows_por_split':            source_rows_por_split,
        'unidades_descartadas':             unidades_descartadas_total,
        'order_cols':                       order_cols,
        'ordered_by_unit_and_cycle':        ordered_by_unit_and_cycle,
        'ordered_by_unit_and_source_order': ordered_by_unit_and_source_order,
    }


def ventanas_de_unidad(X_unidad, target_unidad,
                       window_size=WINDOW_SIZE, stride=STRIDE,
                       tail_policy=None):
    if tail_policy is None:
        tail_policy = TAIL_POLICY
    T, C = X_unidad.shape
    out = []
    if T <= window_size:
        X = np.zeros((window_size, C), dtype='float32')
        X[:T] = X_unidad.astype('float32')
        vtm = np.zeros(window_size, dtype=bool); vtm[:T] = True
        target_win = target_unidad[T - 1]
        out.append((X, vtm, target_win, 0, int(T)))
    else:
        for inicio in range(0, T - window_size + 1, stride):
            fin = inicio + window_size
            X = X_unidad[inicio:fin].astype('float32')
            vtm = np.ones(window_size, dtype=bool)
            target_win = target_unidad[fin - 1]
            out.append((X, vtm, target_win, int(inicio), int(fin)))
        if tail_policy == 'pad':
            ultima_w_end = ((T - window_size) // stride) * stride + window_size
            if ultima_w_end < T:
                resto = T - ultima_w_end
                X = np.zeros((window_size, C), dtype='float32')
                X[:resto] = X_unidad[ultima_w_end:T].astype('float32')
                vtm = np.zeros(window_size, dtype=bool); vtm[:resto] = True
                target_win = target_unidad[T - 1]
                out.append((X, vtm, target_win, int(ultima_w_end), int(T)))
    return out


def normalizar_instancia(X, vtm, eps=1e-6):
    W, C = X.shape
    m = vtm.astype('float32')[:, None]
    n_validos = m.sum()
    if n_validos < 2:
        return (X * m, np.zeros(C, dtype='float32'), np.ones(C, dtype='float32'),
                np.zeros(C, dtype=bool))
    mean = ((X * m).sum(axis=0) / n_validos).astype('float32')
    var  = (((X - mean) ** 2 * m).sum(axis=0) / n_validos).astype('float32')
    canales_const = (var < eps)
    std_raw = np.sqrt(np.maximum(var, 0.0)).astype('float32')
    std_used = np.where(canales_const, 1.0, np.maximum(std_raw, eps)).astype('float32')
    X_norm = (X - mean) / std_used
    X_norm = X_norm * m
    return X_norm.astype('float32'), mean, std_used, canales_const


def patches_C_N_P(X_norm, vtm, patch_size=PATCH_SIZE):
    W, C = X_norm.shape
    assert W % patch_size == 0, f'W={W} no es multiplo de patch_size={patch_size}'
    N = W // patch_size
    patches = X_norm.reshape(N, patch_size, C).transpose(2, 0, 1).astype('float32')
    vtm_patches = vtm.reshape(N, patch_size).any(axis=1)
    valid_patch_mask = np.broadcast_to(vtm_patches, (C, N)).astype(bool).copy()
    return patches, valid_patch_mask


print('Helpers de pipeline definidos (val_fallback, NaN, ordenamiento, ventaneo, normalizacion, patches).')


## 7. Reentrancia segura: hashing + manifest_matches_full + validacion de shards

`pipeline_config_hash` se calcula tras detectar columnas (incluye `signal_cols_hash` y `metadata_cols_hash` sensibles al orden). `semantic_config_hash` excluye `pipeline_version`: util para detectar equivalencia funcional entre v0.4 y v0.5.

`manifest_matches_full` valida hash + config + shards declarados existen + metricas clave coinciden con audit. Si todo eso y `done.flag` existen, el dataset esta cerrado: se recupera la fila del summary desde el manifest sin reprocesar.

`validar_roundtrip_shards` lee el primer `.tar` real escrito con `tarfile` + `np.load`/`json.loads` y verifica shapes, dtypes y que `target_col` no aparece en features.


In [ ]:
import hashlib
import io
import json as _json
import os
import subprocess
import tarfile


def _hash_lista(items):
    """Sensible al orden."""
    return hashlib.sha256(
        _json.dumps(list(items), ensure_ascii=False).encode('utf-8')
    ).hexdigest()[:16]


def construir_pipeline_config(dataset, role, audit_version, columnas):
    config = {
        'dataset':              dataset,
        'role':                 role,
        'audit_version':        audit_version,
        'pipeline_version':     PIPELINE_VERSION,
        'tail_policy':          TAIL_POLICY,
        'window_size':          WINDOW_SIZE,
        'stride':               STRIDE,
        'patch_size':           PATCH_SIZE,
        'mask_ratio':           MASK_RATIO,
        'shard_maxcount':       SHARD_MAXCOUNT,
        'normalization_policy': 'instance_norm_por_ventana_y_canal_ignorando_padding',
        'nan_policy':           f'interpolacion_linear_por_unidad_descarte_si_pct_nan>{MAX_NAN_PCT_UNIDAD}',
        'target_policy':        'ultimo_valor_valido',
        'target_col':           columnas['target_col_seleccionado'],
        'subset_id_col':        columnas.get('subset_id_col'),
        'signal_cols_hash':     _hash_lista(columnas['signal_cols']),
        'metadata_cols_hash':   _hash_lista(columnas['metadata_cols']),
    }
    config_str  = _json.dumps(config, sort_keys=True)
    config_hash = hashlib.sha256(config_str.encode('utf-8')).hexdigest()[:16]
    # semantic hash: excluye pipeline_version (util para comparar v0.4 vs v0.5)
    semantic = dict(config); semantic.pop('pipeline_version', None)
    semantic_str  = _json.dumps(semantic, sort_keys=True)
    semantic_hash = hashlib.sha256(semantic_str.encode('utf-8')).hexdigest()[:16]
    return config, config_hash, semantic_hash


def get_pipeline_code_version():
    v = os.environ.get('PHMD_PIPELINE_CODE_VERSION') or os.environ.get('GIT_COMMIT')
    if v:
        return v, False
    try:
        out = subprocess.check_output(
            ['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'],
            stderr=subprocess.DEVNULL, timeout=5).decode().strip()
        try:
            status = subprocess.check_output(
                ['git', '-C', str(REPO), 'status', '--porcelain'],
                stderr=subprocess.DEVNULL, timeout=5).decode().strip()
            dirty = bool(status)
        except Exception:
            dirty = False
        return (out or None), dirty
    except Exception:
        return None, False


def manifest_matches_full(manifest_path, expected_full, expected_metrics):
    """Devuelve (ok, motivo). True solo si manifest matchea config completa,
    metricas del audit y los shards declarados existen."""
    if not manifest_path.exists():
        return False, 'no existe manifest.json'
    try:
        m = _json.loads(manifest_path.read_text())
    except Exception as e:
        return False, f'manifest no se puede leer: {e}'

    # 1. Config completa
    for k, v in expected_full.items():
        if m.get(k) != v:
            return False, f'campo {k} difiere: manifest={m.get(k)!r} expected={v!r}'

    # 2. Metricas clave del manifest deben coincidir con audit
    for k, v in expected_metrics.items():
        mv = m.get(k)
        if mv is None or v is None:
            continue
        # Comparacion tolerante para floats
        if isinstance(v, float):
            if abs(float(mv) - v) > 1e-3:
                return False, f'metrica {k} difiere: manifest={mv} audit={v}'
        else:
            if mv != v:
                return False, f'metrica {k} difiere: manifest={mv!r} audit={v!r}'

    # 3. Shards declarados existen en disco
    shard_paths = m.get('shard_paths') or {}
    for split, paths in shard_paths.items():
        for p in paths:
            if not Path(p).exists():
                return False, f'shard declarado no existe: {p}'

    # 4. manifest no debe tener done=True
    if m.get('done') is True:
        return False, 'manifest contiene done=True (no permitido en v0.5)'

    return True, 'match'


def reconstruir_fila_summary_desde_manifest(manifest_path, run_mode):
    """Recupera una fila para processed_summary a partir del manifest existente.
    Util cuando SKIP_VALID: no reprocesamos pero queremos la fila en el CSV."""
    m = _json.loads(manifest_path.read_text())
    done_flag = manifest_path.parent / 'done.flag'
    n_units_total = sum((m.get('n_units_por_split') or {}).values()) or m.get('n_units_total')
    fila = {
        'dataset':              m.get('dataset'),
        'role':                 m.get('role'),
        'client':               m.get('client'),
        'dominio':              m.get('dominio'),
        'evaluation_tier':      m.get('evaluation_tier'),
        'status':               'SKIP_VALID',
        'audit_version':        m.get('audit_version'),
        'pipeline_version':     m.get('pipeline_version'),
        'pipeline_code_version': m.get('pipeline_code_version'),
        'pipeline_config_hash': m.get('pipeline_config_hash'),
        'run_mode':             run_mode,
        'tail_policy':          m.get('tail_policy'),
        'n_channels':           m.get('n_channels'),
        'n_units_total':        n_units_total,
        'n_windows_total':      m.get('n_windows_total'),
        'n_temporal_patches':   m.get('n_temporal_patches'),
        'n_channel_patches':    m.get('n_channel_patches'),
        'n_dense_temporal_patches': m.get('n_dense_temporal_patches'),
        'n_dense_channel_patches':  m.get('n_dense_channel_patches'),
        'invalid_patch_ratio':  m.get('invalid_patch_ratio'),
        'dense_vs_valid_ratio': m.get('dense_vs_valid_ratio'),
        'padding_ratio':        m.get('padding_ratio'),
        'source_rows_total':    (m.get('row_accounting') or {}).get('source_rows_total'),
        'source_rows_covered_unique': (m.get('row_accounting') or {}).get('source_rows_covered_unique'),
        'dropped_tail_rows':    (m.get('row_accounting') or {}).get('dropped_tail_rows_due_to_windowing'),
        'estimated_n_shards':   m.get('estimated_n_shards_audit'),
        'actual_n_shards':      m.get('actual_n_shards'),
        'actual_size_mb_total': m.get('actual_size_mb_total'),
        'warnings_count':       len(m.get('warnings') or []),
        'anti_leakage_status':  'pass',
        'roundtrip_status':     (m.get('roundtrip_validation') or {}).get('result', 'unknown'),
        'manifest_path':        str(manifest_path),
        'done_flag_path':       str(done_flag) if done_flag.exists() else None,
    }
    return fila


def validar_roundtrip_shards(shard_paths_por_split, expected_C, expected_N, expected_P,
                              dataset, target_col, signal_cols):
    """Lee el primer .tar real y valida estructura + shapes."""
    first_shard = None
    first_split = None
    for sn, paths in shard_paths_por_split.items():
        if paths:
            first_shard = paths[0]
            first_split = sn
            break
    if first_shard is None:
        return {'result': 'skip', 'note': 'no hay shards reales'}

    expected_exts = {'patches.npy', 'valid_time_mask.npy', 'valid_patch_mask.npy',
                     'mean.npy', 'std_used.npy', 'canales_constantes_mask.npy',
                     'target.json', 'meta.json'}
    samples_por_key = {}
    try:
        with tarfile.open(first_shard, 'r') as tar:
            for m in tar.getmembers():
                if not m.isfile(): continue
                # nombres tipo "<key>.<suffix>"; suffix puede tener punto (e.g. ".npy")
                base = m.name
                if '.' not in base:
                    continue
                key, ext = base.split('.', 1)
                # leer maximo 2 muestras
                if len(samples_por_key) >= 2 and key not in samples_por_key:
                    continue
                fobj = tar.extractfile(m)
                if fobj is None:
                    continue
                data = fobj.read()
                if ext.endswith('npy'):
                    arr = np.load(io.BytesIO(data), allow_pickle=False)
                    samples_por_key.setdefault(key, {})[ext] = arr
                elif ext.endswith('json'):
                    obj = _json.loads(data.decode('utf-8'))
                    samples_por_key.setdefault(key, {})[ext] = obj
    except Exception as e:
        return {'result': 'fail', 'note': f'no se pudo abrir/leer tar: {e}', 'tar': str(first_shard)}

    if not samples_por_key:
        return {'result': 'fail', 'note': 'tar sin samples', 'tar': str(first_shard)}

    primer_key = next(iter(samples_por_key))
    sample = samples_por_key[primer_key]
    presentes = set(sample.keys())
    faltantes = expected_exts - presentes
    if faltantes:
        return {'result': 'fail', 'note': f'faltan claves: {sorted(faltantes)}',
                'tar': str(first_shard), 'presentes': sorted(presentes)}

    patches = sample['patches.npy']
    if patches.shape != (expected_C, expected_N, expected_P):
        return {'result': 'fail',
                'note': f'patches.shape {patches.shape} != ({expected_C},{expected_N},{expected_P})',
                'tar': str(first_shard)}
    if patches.dtype != np.float32:
        return {'result': 'fail', 'note': f'patches.dtype {patches.dtype} != float32'}

    vtm = sample['valid_time_mask.npy']
    if vtm.shape != (WINDOW_SIZE,):
        return {'result': 'fail', 'note': f'vtm.shape {vtm.shape} != ({WINDOW_SIZE},)'}
    if vtm.dtype != bool:
        return {'result': 'fail', 'note': f'vtm.dtype {vtm.dtype} != bool'}

    vpm = sample['valid_patch_mask.npy']
    if vpm.shape != (expected_C, expected_N):
        return {'result': 'fail', 'note': f'vpm.shape {vpm.shape} != ({expected_C},{expected_N})'}
    if vpm.dtype != bool:
        return {'result': 'fail', 'note': f'vpm.dtype {vpm.dtype} != bool'}

    target = sample['target.json']
    if 'target_window' not in target:
        return {'result': 'fail', 'note': 'target.json sin target_window'}
    if target.get('target_col') == target_col and target_col in signal_cols:
        return {'result': 'fail', 'note': f'target_col {target_col} esta en signal_cols'}

    meta = sample['meta.json']
    if meta.get('dataset') != dataset:
        return {'result': 'fail',
                'note': f'meta.dataset {meta.get("dataset")!r} != {dataset!r}'}

    return {
        'result':                'pass',
        'tar':                   str(first_shard),
        'split':                 first_split,
        'samples_leidos':        len(samples_por_key),
        'patches_shape':         list(patches.shape),
        'meta_dataset':          meta.get('dataset'),
        'meta_role':             meta.get('role'),
        'meta_pipeline_version': meta.get('pipeline_version'),
    }


print('Helpers de hashing + manifest_matches_full + validar_roundtrip_shards definidos.')


## 8. Anti-leakage por dataset

Mismos 10 checks del piloto, generalizados. `documented_expected_for_dataset` sustituye al `_for_CMAPSS` cuando hay reuso legitimo de `unit_id` entre splits.


In [ ]:
def construir_anti_leakage(dataset, splits, columnas, primer_sample, batch,
                          val_fallback_meta, order_info, effective_mask_ratio):
    trajs = {sn: set(splits[sn]['trajectory_id'].unique())
             for sn in splits if splits[sn] is not None}
    assets = {sn: set(splits[sn]['asset_key_without_split'].unique())
              for sn in splits if splits[sn] is not None}

    inter_tv = (trajs.get('train', set()) & trajs.get('val',  set()))
    inter_tt = (trajs.get('train', set()) & trajs.get('test', set()))
    inter_vt = (trajs.get('val',   set()) & trajs.get('test', set()))

    asset_reused_train_test = (assets.get('train', set()) & assets.get('test', set()))
    asset_reused_train_val  = (assets.get('train', set()) & assets.get('val',  set()))

    target_col = columnas['target_col_seleccionado']
    signal_cols = columnas['signal_cols']

    return {
        'dataset':                  dataset,
        'pipeline_version':         PIPELINE_VERSION,
        'trajectory_id_policy':     'dataset__subset-<slug>__split-<split>__unit-<slug>',
        'asset_key_without_split_policy': 'dataset__subset-<slug>__unit-<slug>',
        'checks': {
            'no_trajectory_id_shared_across_splits': {
                'result': 'pass' if (len(inter_tv) == len(inter_tt) == len(inter_vt) == 0) else 'fail',
                'n_train_trajectories': len(trajs.get('train', set())),
                'n_val_trajectories':   len(trajs.get('val', set())),
                'n_test_trajectories':  len(trajs.get('test', set())),
                'intersection_train_val':  len(inter_tv),
                'intersection_train_test': len(inter_tt),
                'intersection_val_test':   len(inter_vt),
            },
            'raw_unit_ids_reused_across_original_splits': {
                'result': ('documented_expected_for_dataset'
                           if (len(asset_reused_train_test) > 0 or len(asset_reused_train_val) > 0)
                           else 'pass'),
                'asset_key_reused_train_test': len(asset_reused_train_test),
                'asset_key_reused_train_val':  len(asset_reused_train_val),
            },
            'split_applied_before_windowing': {
                'result': 'pass',
                'val_fallback': val_fallback_meta,
            },
            'data_ordered_by_unit_and_time': {
                'result': ('pass' if order_info['ordered_by_unit_and_cycle']
                           else 'pass_without_time_col'),
                'order_cols': order_info['order_cols'],
            },
            'no_test_stats_used_in_normalization': {
                'result': 'pass',
            },
            'patches_shape_contract': {
                'result': 'pass',
                'shape':  list(primer_sample['patches'].shape),
                'dtype':  str(primer_sample['patches'].dtype),
            },
            'batch_shape_contract': {
                'result': 'pass',
                'shape':  list(batch['patches'].shape),
                'dtype':  str(batch['patches'].dtype),
            },
            'masks_dtype_bool': {
                'result': 'pass',
                'valid_time_mask':  str(batch['valid_time_mask'].dtype),
                'valid_patch_mask': str(batch['valid_patch_mask'].dtype),
                'ssl_mask':         str(batch['ssl_mask'].dtype),
            },
            'target_separated_from_features': {
                'result': 'pass' if (target_col is None or target_col not in signal_cols) else 'fail',
                'target_col': target_col,
                'n_signal_cols': len(signal_cols),
            },
            'effective_mask_ratio_within_tolerance': {
                'result': 'pass' if abs(effective_mask_ratio - MASK_RATIO) < 0.05 else 'fail',
                'effective_mask_ratio': round(effective_mask_ratio, 4),
                'target_mask_ratio':    MASK_RATIO,
                'tolerance':            0.05,
            },
        },
    }


print('Helper anti_leakage definido.')


## 9. `procesar_dataset` con streaming + staging (sin acumular samples)

Refactor importante respecto al piloto: el piloto acumulaba TODOS los samples en memoria. Para datasets grandes (PHM10, IMS, XJTU-SY, NCMAPSS, HIRFNASA15) eso es OOM. En v0.5 cada ventana se serializa y se escribe al `ShardWriter` inmediatamente. Solo mantenemos:

- contadores incrementales (n_windows, padding, patches validos)
- `primer_sample` para validar contrato
- `batch_buffer` (hasta 8 samples del primer split no vacio) para validar `(B,C,N,P)` y `mask_ratio` efectivo
- `warnings_aggregated` por canal (std_zero)

Flujo `procesar_dataset(name, fila_audit, ...)`:

1. cargar phmd
2. detectar columnas + validar contra audit + asserts de columnas
3. construir `pipeline_config_hash` y `semantic_config_hash`
4. **validar manifest existente (full match)**: si todo coincide y `done.flag` existe y `RUN_MODE != "full"` o no se fuerza, devuelve `SKIP_VALID` con la fila reconstruida
5. crear `staging_dir = _staging_v0_5/<dataset>__<run_id>/`
6. construir `trajectory_id`
7. val_fallback si phmd no entrega val
8. assert anti-leakage 1 (trajectory_id disjunto entre splits)
9. limpiar NaN + ordenar
10. **bucle por trayectoria en streaming**: ventanear, normalizar, patchear, serializar, escribir al ShardWriter; acumular `primer_sample`, `batch_buffer`, contadores y warnings
11. validar contrato del primer sample + batch + mask_ratio
12. construir `anti_leakage_report` con primer_sample y batch
13. roundtrip real de los `.tar` escritos
14. asserts duros contra audit
15. escribir `anti_leakage_report.json`, `manifest.json` (sin `done=True`), `warnings_detail.jsonl` en staging
16. **promover staging** a `final_dir` (canonical o `_smoke_v0_5/`):
    - si existe `final_dir` valido (manifest + done.flag), moverlo a `_backup_v0_5_<run_id>/`
    - mover staging a `final_dir`
    - escribir `done.flag` en final_dir (ULTIMO artefacto)
17. devolver fila summary


In [ ]:
import io
from datetime import datetime, timezone

import torch
from torch.utils.data import IterableDataset, DataLoader
import webdataset as wds


def generar_ssl_mask(valid_patch_mask, mask_ratio, rng):
    C, N = valid_patch_mask.shape
    ssl_mask = np.zeros((C, N), dtype=bool)
    for c in range(C):
        validos = np.where(valid_patch_mask[c])[0]
        if len(validos) == 0:
            continue
        k = max(1, int(round(len(validos) * mask_ratio)))
        elegidos = rng.choice(validos, size=k, replace=False)
        ssl_mask[c, elegidos] = True
    return ssl_mask


def serializar_sample(s, idx, dataset, role, evaluation_tier, client_name, dominio):
    """Empaqueta un sample en un dict para ShardWriter (.tar)."""
    key = f'{s["trajectory_id"]}_w{idx:06d}'
    out = {'__key__': key}
    for k_arr in ['patches', 'valid_time_mask', 'valid_patch_mask',
                  'mean', 'std_used', 'canales_constantes_mask']:
        buf = io.BytesIO()
        np.save(buf, s[k_arr], allow_pickle=False)
        out[f'{k_arr}.npy'] = buf.getvalue()
    out['target.json'] = _json.dumps(to_jsonable({
        'target_window':  s['target_window'],
        'target_col':     s['target_col'],
        'target_policy':  'ultimo_valor_valido',
        'target_warning': s['target_warning'],
    }), allow_nan=False).encode('utf-8')
    out['meta.json'] = _json.dumps(to_jsonable({
        'dataset':                  dataset,
        'role':                     role,
        'evaluation_tier':          evaluation_tier,
        'client':                   client_name,
        'dominio':                  dominio,
        'pipeline_version':         PIPELINE_VERSION,
        'subset_id':                s.get('subset_id'),
        'split_origen':             s['split_origen'],
        'unit_id':                  s.get('unit_id'),
        'trajectory_id':            s['trajectory_id'],
        'unit_global_id':           s['trajectory_id'],
        'asset_key_without_split':  s['asset_key_without_split'],
        'idx_window':               s['idx_window'],
        'window_start':             s['window_start'],
        'window_end':               s['window_end'],
        'window_size':              WINDOW_SIZE,
        'stride':                   STRIDE,
        'patch_size':               PATCH_SIZE,
    }), allow_nan=False).encode('utf-8')
    return out


class _VentanasIterFromBuffer(IterableDataset):
    """IterableDataset que devuelve los samples del batch_buffer (en memoria)."""
    def __init__(self, samples, mask_ratio, seed):
        super().__init__()
        self.samples = samples
        self.mask_ratio = mask_ratio
        self.seed = seed
    def __iter__(self):
        rng = np.random.default_rng(self.seed)
        for s in self.samples:
            ssl = generar_ssl_mask(s['valid_patch_mask'], self.mask_ratio, rng)
            yield {
                'patches':          torch.from_numpy(s['patches']),
                'valid_time_mask':  torch.from_numpy(s['valid_time_mask']),
                'valid_patch_mask': torch.from_numpy(s['valid_patch_mask']),
                'ssl_mask':         torch.from_numpy(ssl),
                'mean':             torch.from_numpy(s['mean']),
                'std':              torch.from_numpy(s['std_used']),
                'target':           torch.tensor(float(s['target_window']), dtype=torch.float32),
            }


def _collate_buffer(batch):
    return {k: torch.stack([b[k] for b in batch]) for k in batch[0]}


print('Helpers de serializacion + ShardWriter wrapper definidos.')


In [ ]:
def procesar_dataset(nombre, fila_audit, log_fn, run_mode, run_id,
                     write_canonical, processed_root_canonical, processed_root_smoke,
                     dataset_to_client, force_reprocess):
    """Procesa un dataset en streaming + staging. Devuelve (status, fila_summary, info)."""
    role            = fila_audit['role']
    evaluation_tier = fila_audit.get('evaluation_tier') if pd.notna(fila_audit.get('evaluation_tier')) else None
    dominio         = fila_audit.get('dominio') if pd.notna(fila_audit.get('dominio')) else None
    client_name     = dataset_to_client.get(nombre) if role == 'PRETRAIN_SOURCE' else None
    audit_version   = str(fila_audit['audit_version'])

    # Lecturas del audit
    audit_n_canales = int(fila_audit['n_canales'])
    audit_n_windows = int(fila_audit['n_windows'])
    audit_pad_ratio = float(fila_audit['padding_ratio'])
    audit_n_temporal_patches = int(fila_audit['n_temporal_patches'])
    audit_n_channel_patches_valid = int(fila_audit['n_channel_patches'])
    audit_n_dense_temporal_patches = int(fila_audit['n_dense_temporal_patches'])
    audit_n_dense_channel_patches  = int(fila_audit['n_dense_channel_patches'])
    audit_invalid_patch_ratio      = float(fila_audit['invalid_patch_ratio'])
    audit_dense_vs_valid_ratio     = float(fila_audit['dense_vs_valid_ratio'])
    audit_estimated_n_shards       = int(fila_audit['estimated_n_shards'])
    audit_n_filas_total            = int(fila_audit['n_filas_total'])
    audit_n_unidades_total         = int(fila_audit['n_unidades_total'])
    audit_target_warning = None
    if pd.notna(fila_audit.get('rul_min')) and float(fila_audit['rul_min']) < 0:
        if fila_audit.get('tipo_target') in ('rul', 'regression_or_rul'):
            audit_target_warning = 'rul_negative_values'
        elif fila_audit.get('tipo_target') == 'regression':
            audit_target_warning = 'negative_target_values'

    # ---------- Rutas: staging y final ----------
    if run_mode == 'smoke':
        final_root = processed_root_smoke
    else:
        final_root = processed_root_canonical
    final_dir   = final_root / nombre
    staging_dir = STAGING_ROOT / f'{nombre}__{run_id}'
    if staging_dir.exists():
        shutil.rmtree(staging_dir, ignore_errors=True)
    staging_dir.mkdir(parents=True, exist_ok=True)

    # ---------- 1. Cargar PHMD ----------
    log_fn(f'  cargando {nombre}...')
    ds, splits, ficheros = cargar_dataset(nombre, verbose=False)

    try:
        # ---------- 2. Detectar columnas + validar contra audit ----------
        columnas = detectar_columnas(ds, splits, nombre)
        validar_columnas_contra_audit(columnas, fila_audit, nombre)
        n_canales = len(columnas['signal_cols'])

        # ---------- 3. Hash + semantic_hash ----------
        pipeline_config, pipeline_config_hash, semantic_config_hash = \
            construir_pipeline_config(nombre, role, audit_version, columnas)

        # ---------- 4. Reentrancia segura (manifest existente full match) ----------
        skip = False
        skip_motivo = None
        if not force_reprocess and final_dir.exists():
            expected_full = {
                'dataset':              nombre,
                'audit_version':        audit_version,
                'pipeline_version':     PIPELINE_VERSION,
                'tail_policy':          TAIL_POLICY,
                'window_size':          WINDOW_SIZE,
                'stride':               STRIDE,
                'patch_size':           PATCH_SIZE,
                'mask_ratio':           MASK_RATIO,
                'shard_maxcount':       SHARD_MAXCOUNT,
                'pipeline_config_hash': pipeline_config_hash,
            }
            expected_metrics = {
                'n_windows_total':         audit_n_windows,
                'n_temporal_patches':      audit_n_temporal_patches,
                'n_channel_patches':       audit_n_channel_patches_valid,
                'n_dense_temporal_patches': audit_n_dense_temporal_patches,
                'n_dense_channel_patches': audit_n_dense_channel_patches,
                'padding_ratio':           round(audit_pad_ratio, 4),
            }
            manifest_path_final = final_dir / 'manifest.json'
            done_flag_final     = final_dir / 'done.flag'
            ok, motivo = manifest_matches_full(manifest_path_final, expected_full, expected_metrics)
            if ok and done_flag_final.exists():
                skip = True
                skip_motivo = f'manifest full match + done.flag ({motivo})'

        if skip:
            log_fn(f'  SKIP_VALID {nombre}: {skip_motivo}')
            fila = reconstruir_fila_summary_desde_manifest(final_dir / 'manifest.json', run_mode)
            # limpia staging vacio
            shutil.rmtree(staging_dir, ignore_errors=True)
            return 'SKIP_VALID', fila, {'motivo': skip_motivo}

        # ---------- 5. trajectory_id ----------
        splits = construir_trajectory_id(splits, columnas, nombre)

        # ---------- 6. val_fallback ----------
        splits, val_fallback_meta = aplicar_val_fallback_si_falta(splits, columnas, nombre)

        # ---------- 7. Anti-leakage check temprano ----------
        trajs_pre = {sn: set(splits[sn]['trajectory_id'].unique())
                     for sn in splits if splits[sn] is not None}
        assert (len(trajs_pre.get('train', set()) & trajs_pre.get('val', set())) == 0 and
                len(trajs_pre.get('train', set()) & trajs_pre.get('test', set())) == 0 and
                len(trajs_pre.get('val', set())  & trajs_pre.get('test', set())) == 0), \
            f'{nombre}: trajectory_id solapa entre splits'

        # ---------- 8. NaN + ordenamiento ----------
        splits, order_info = limpiar_nans_y_ordenar(splits, columnas)
        source_rows_total     = order_info['source_rows_total']
        source_rows_por_split = order_info['source_rows_por_split']

        # ---------- 9. Bucle por trayectoria EN STREAMING ----------
        unit_col      = columnas['unit_col']
        subset_id_col = columnas.get('subset_id_col')
        target_col    = columnas['target_col_seleccionado']
        signal_cols   = columnas['signal_cols']

        # Acumuladores ligeros (no listas de samples)
        n_windows_por_split  = {sn: 0 for sn in splits.keys()}
        n_valid_patches_por_split = {sn: 0 for sn in splits.keys()}
        n_channel_patches_por_split = {sn: 0 for sn in splits.keys()}
        n_trajectories_por_split = {sn: 0 for sn in splits.keys()}
        n_units_por_split    = {sn: 0 for sn in splits.keys()}
        total_muestras_padding = 0
        valid_time_positions_across_windows = 0
        dropped_tail_rows = 0
        warnings_pipeline = []
        std_zero_por_canal = {}
        _traj_con_std_zero = set()

        # Mini buffers para validacion
        primer_sample = None
        batch_buffer  = []     # max 8 samples del primer split no vacio
        primer_split_no_vacio = None
        BUFFER_MAX = 8

        # Path de shards
        shards_paths_por_split = {}
        actual_size_mb_por_split = {}

        # Abrir ShardWriter por split y procesar streaming
        for split_name, df in splits.items():
            # Contar por trajectory_id (no unit_col crudo): trajectory_id incluye
            # split + subset_id + unit, por lo que distingue subsets de CMAPSS y
            # coincide con n_unidades_total del audit (alias n_trayectorias_total).
            # Con unit_col crudo, CMAPSS reportaba 629 vs 1416 correcto por reuso
            # de unit_id entre los 4 FDs.
            if 'trajectory_id' in df.columns:
                n_units_por_split[split_name] = int(df['trajectory_id'].nunique())
            elif unit_col in df.columns:
                n_units_por_split[split_name] = int(df[unit_col].nunique())
            else:
                n_units_por_split[split_name] = 0
            split_dir = staging_dir / split_name
            split_dir.mkdir(parents=True, exist_ok=True)
            pattern = str(split_dir / f'{nombre}-{split_name}-%06d.tar')
            with wds.ShardWriter(pattern, maxcount=SHARD_MAXCOUNT) as sink:
                for traj_id, sub in df.groupby('trajectory_id', sort=False):
                    n_trajectories_por_split[split_name] += 1
                    X_unidad = sub[signal_cols].to_numpy() if signal_cols else np.empty((len(sub), 0))
                    target_unidad = sub[target_col].to_numpy() if target_col else np.zeros(len(sub))
                    T_unidad = X_unidad.shape[0]
                    if T_unidad == 0 or X_unidad.shape[1] == 0:
                        continue
                    ventanas = ventanas_de_unidad(X_unidad, target_unidad, tail_policy=TAIL_POLICY)
                    if not ventanas:
                        continue
                    ultimo_w_end = max(int(w_end) for (_, _, _, _, w_end) in ventanas)
                    if T_unidad > ultimo_w_end:
                        dropped_tail_rows += (T_unidad - ultimo_w_end)
                    for idx_w, (X_win, vtm, target_win, w_start, w_end) in enumerate(ventanas):
                        X_norm, mean_c, std_used, const_mask = normalizar_instancia(X_win, vtm)
                        if const_mask.any() and traj_id not in _traj_con_std_zero:
                            _traj_con_std_zero.add(traj_id)
                            canales_const = [signal_cols[c] for c in np.where(const_mask)[0]]
                            for cc in canales_const:
                                std_zero_por_canal.setdefault(cc, set()).add(traj_id)
                            warnings_pipeline.append({
                                'tipo': 'std_zero', 'trajectory_id': traj_id,
                                'canales_constantes': canales_const,
                            })
                        patches, vpm = patches_C_N_P(X_norm, vtm)

                        # Acumuladores
                        n_windows_por_split[split_name] += 1
                        valid_time_positions_across_windows += int(vtm.sum())
                        total_muestras_padding += int((~vtm).sum())
                        n_valid_patches_por_split[split_name]   += int(vpm[0].sum())
                        n_channel_patches_por_split[split_name] += int(vpm.sum())

                        sample = {
                            'patches':                 patches,
                            'valid_time_mask':         vtm,
                            'valid_patch_mask':        vpm,
                            'mean':                    mean_c,
                            'std_used':                std_used,
                            'canales_constantes_mask': const_mask,
                            'target_window':           float(target_win),
                            'target_col':              target_col,
                            'target_warning':          audit_target_warning,
                            'trajectory_id':           traj_id,
                            'asset_key_without_split': sub['asset_key_without_split'].iloc[0],
                            'idx_window':              idx_w,
                            'split_origen':            split_name,
                            'subset_id':               sub[subset_id_col].iloc[0] if subset_id_col else None,
                            'unit_id':                 sub[unit_col].iloc[0],
                            'window_start':            w_start,
                            'window_end':              w_end,
                        }

                        # Guardar primer_sample y poblar batch_buffer
                        if primer_sample is None:
                            primer_sample = sample
                        if primer_split_no_vacio is None:
                            primer_split_no_vacio = split_name
                        if split_name == primer_split_no_vacio and len(batch_buffer) < BUFFER_MAX:
                            # copia para no contaminar al sink
                            batch_buffer.append({k: (v.copy() if isinstance(v, np.ndarray) else v)
                                                  for k, v in sample.items()})

                        # Serializar y escribir AHORA al ShardWriter
                        rec = serializar_sample(sample, idx_w, nombre, role,
                                                 evaluation_tier, client_name, dominio)
                        sink.write(rec)
            # Listar shards reales escritos por split
            shards = sorted(str(p) for p in split_dir.glob('*.tar'))
            shards_paths_por_split[split_name] = shards
            actual_size_mb_por_split[split_name] = round(
                sum(Path(p).stat().st_size for p in shards) / 1e6, 2)

        # ---------- 10. Metricas finales ----------
        n_windows_total       = sum(n_windows_por_split.values())
        n_temporal_patches    = sum(n_valid_patches_por_split.values())
        n_channel_patches_valid = sum(n_channel_patches_por_split.values())
        padding_ratio_actual  = (total_muestras_padding / (n_windows_total * WINDOW_SIZE)
                                  if n_windows_total else 0.0)
        pilot_dense_temporal_patches = n_windows_total * N_PATCHES
        pilot_dense_channel_patches  = pilot_dense_temporal_patches * n_canales
        invalid_patch_ratio = round(1.0 - (n_channel_patches_valid / pilot_dense_channel_patches), 4) \
            if pilot_dense_channel_patches else None
        dense_vs_valid_ratio = round(pilot_dense_channel_patches / n_channel_patches_valid, 4) \
            if n_channel_patches_valid else None
        actual_n_shards = sum(len(v) for v in shards_paths_por_split.values())
        actual_size_mb_total = round(sum(actual_size_mb_por_split.values()), 2)
        source_rows_covered_unique = source_rows_total - dropped_tail_rows

        # ---------- 11. Validacion contrato + batch + mask_ratio ----------
        assert primer_sample is not None, f'{nombre}: no se genero ningun sample'
        assert primer_sample['patches'].shape == (n_canales, N_PATCHES, PATCH_SIZE)
        assert primer_sample['patches'].dtype == np.float32

        ds_buf = _VentanasIterFromBuffer(batch_buffer, MASK_RATIO, seed=SEED)
        dl = DataLoader(ds_buf, batch_size=min(8, len(batch_buffer)),
                        collate_fn=_collate_buffer, num_workers=0)
        batch = next(iter(dl))
        valid_total  = int(batch['valid_patch_mask'].sum().item())
        masked_total = int(batch['ssl_mask'].sum().item())
        effective_mask_ratio = (masked_total / valid_total) if valid_total else 0.0

        # ---------- 12. Anti-leakage report ----------
        anti_leakage = construir_anti_leakage(
            nombre, splits, columnas, primer_sample, batch,
            val_fallback_meta, order_info, effective_mask_ratio)
        results_ok = ('pass', 'pass_without_time_col', 'documented_expected_for_dataset')
        all_pass_al = all(c['result'] in results_ok for c in anti_leakage['checks'].values())
        assert all_pass_al, f'{nombre}: anti-leakage FAIL: {anti_leakage["checks"]}'

        # ---------- 13. Roundtrip REAL leyendo .tar ----------
        roundtrip_info = validar_roundtrip_shards(
            shards_paths_por_split, n_canales, N_PATCHES, PATCH_SIZE,
            nombre, target_col, signal_cols)
        if roundtrip_info.get('result') != 'pass':
            raise RuntimeError(f'{nombre}: roundtrip FAIL: {roundtrip_info}')

        # ---------- 14. ASSERTS DUROS contra audit (antes de escribir manifest) ----------
        assert n_windows_total == audit_n_windows, (
            f'{nombre}: n_windows piloto={n_windows_total} vs audit={audit_n_windows}')
        assert round(padding_ratio_actual, 4) == round(audit_pad_ratio, 4), (
            f'{nombre}: padding {padding_ratio_actual} vs audit {audit_pad_ratio}')
        assert n_temporal_patches == audit_n_temporal_patches, (
            f'{nombre}: temporal_patches {n_temporal_patches} vs audit {audit_n_temporal_patches}')
        assert n_channel_patches_valid == audit_n_channel_patches_valid, (
            f'{nombre}: channel_patches {n_channel_patches_valid} vs audit {audit_n_channel_patches_valid}')
        assert pilot_dense_temporal_patches == audit_n_dense_temporal_patches, (
            f'{nombre}: dense_temporal {pilot_dense_temporal_patches} vs audit {audit_n_dense_temporal_patches}')
        assert pilot_dense_channel_patches == audit_n_dense_channel_patches, (
            f'{nombre}: dense_channel {pilot_dense_channel_patches} vs audit {audit_n_dense_channel_patches}')
        assert audit_estimated_n_shards == actual_n_shards, (
            f'{nombre}: shards audit={audit_estimated_n_shards} vs actual={actual_n_shards}')
        assert source_rows_total == audit_n_filas_total, (
            f'{nombre}: source_rows_total {source_rows_total} vs audit {audit_n_filas_total}')
        # Detecta divergencias semanticas de unidades (e.g. unit_col crudo vs
        # trajectory_id en CMAPSS y otros con subset_id_col).
        n_units_total_actual = sum(n_units_por_split.values())
        assert n_units_total_actual == audit_n_unidades_total, (
            f'{nombre}: n_units piloto={n_units_total_actual} vs '
            f'audit={audit_n_unidades_total} (deberian coincidir si se cuenta '
            f'por trajectory_id, no por unit_col crudo)')
        if TAIL_POLICY == 'pad':
            assert dropped_tail_rows == 0, f'{nombre}: pad da dropped={dropped_tail_rows}'
            assert source_rows_covered_unique == source_rows_total, (
                f'{nombre}: covered_unique {source_rows_covered_unique} vs total {source_rows_total}')

        # ---------- 15. Escribir artefactos en STAGING ----------
        pipeline_code_version, git_dirty = get_pipeline_code_version()

        warnings_aggregated_std_zero = [
            {
                'tipo':                     'std_zero_canal_constante',
                'canal':                    canal,
                'n_trayectorias_afectadas': len(tr),
            }
            for canal, tr in sorted(std_zero_por_canal.items(), key=lambda kv: -len(kv[1]))
        ]

        manifest = {
            'manifest_schema_version':  MANIFEST_SCHEMA_VERSION,
            'dataset':                  nombre,
            'role':                     role,
            'client':                   client_name,
            'dominio':                  dominio,
            'evaluation_tier':          evaluation_tier,
            'audit_version':            audit_version,
            'pipeline_version':         PIPELINE_VERSION,
            'pipeline_code_version':    pipeline_code_version,
            'git_dirty':                bool(git_dirty),
            'generated_at':             datetime.now(timezone.utc).isoformat(timespec='seconds'),
            'run_id':                   run_id,
            'run_mode':                 run_mode,
            'pipeline_config_hash':     pipeline_config_hash,
            'semantic_config_hash':     semantic_config_hash,
            'pipeline_config':          pipeline_config,
            'tail_policy':              TAIL_POLICY,
            'window_size':              WINDOW_SIZE,
            'stride':                   STRIDE,
            'patch_size':               PATCH_SIZE,
            'n_patches':                N_PATCHES,
            'mask_ratio':               MASK_RATIO,
            'effective_mask_ratio_check_batch': round(effective_mask_ratio, 4),
            'shard_maxcount':           SHARD_MAXCOUNT,
            'normalization_policy':     'instance_norm_por_ventana_y_canal_ignorando_padding',
            'nan_policy':               f'interpolacion_linear_por_unidad_descarte_si_pct_nan>{MAX_NAN_PCT_UNIDAD}',
            'batching_policy':          'por_dataset',
            'unit_col':                 columnas['unit_col'],
            'time_col':                 columnas.get('time_col'),
            'subset_id_col':            columnas.get('subset_id_col'),
            'target_col':               target_col,
            'target_candidates':        columnas.get('target_candidates') or [],
            'target_policy':            'ultimo_valor_valido',
            'target_warning':           audit_target_warning,
            'signal_cols':              columnas['signal_cols'],
            'metadata_cols':            columnas['metadata_cols'],
            'signal_cols_hash':         pipeline_config['signal_cols_hash'],
            'metadata_cols_hash':       pipeline_config['metadata_cols_hash'],
            'unit_global_id_policy':    'trajectory_id (incluye split + subset_id + unit)',
            'val_fallback':             val_fallback_meta,
            'order_info':               order_info,
            'n_channels':               n_canales,
            'n_units_total':            sum(n_units_por_split.values()),
            'n_units_por_split':        n_units_por_split,
            'n_windows_total':          n_windows_total,
            'n_windows_por_split':      n_windows_por_split,
            'n_temporal_patches':       n_temporal_patches,
            'n_temporal_patches_por_split': n_valid_patches_por_split,
            'n_channel_patches':        n_channel_patches_valid,
            'n_channel_patches_por_split': n_channel_patches_por_split,
            'n_dense_temporal_patches': pilot_dense_temporal_patches,
            'n_dense_channel_patches':  pilot_dense_channel_patches,
            'invalid_patch_ratio':      invalid_patch_ratio,
            'dense_vs_valid_ratio':     dense_vs_valid_ratio,
            'padding_ratio':            round(padding_ratio_actual, 4),
            'source_rows_total':        source_rows_total,
            'source_rows_covered_unique': source_rows_covered_unique,
            'dropped_tail_rows':        dropped_tail_rows,
            'valid_time_positions_across_windows': valid_time_positions_across_windows,
            'row_accounting': {
                'source_rows_total':                  source_rows_total,
                'source_rows_por_split':              source_rows_por_split,
                'source_rows_covered_unique':         source_rows_covered_unique,
                'valid_time_positions_across_windows': valid_time_positions_across_windows,
                'dropped_tail_rows_due_to_windowing': dropped_tail_rows,
                'audit_n_filas_total':                audit_n_filas_total,
            },
            'estimated_n_shards':       audit_estimated_n_shards,
            'estimated_n_shards_audit': audit_estimated_n_shards,
            'actual_n_shards':          actual_n_shards,
            'shard_paths':              shards_paths_por_split,
            'actual_size_mb_por_split': actual_size_mb_por_split,
            'actual_size_mb_total':     actual_size_mb_total,
            'anti_leakage_checks':      {k: v['result'] for k, v in anti_leakage['checks'].items()},
            'roundtrip_validation':     roundtrip_info,
            'normalization_stats_saved': True,
            'warnings':                 warnings_pipeline,
            'warnings_aggregated':      {'std_zero_por_canal': warnings_aggregated_std_zero},
        }
        manifest = to_jsonable(manifest)

        # Escritura atomica en staging
        for tmp_name, contenido in [
            ('anti_leakage_report.json', to_jsonable(anti_leakage)),
            ('manifest.json',            manifest),
        ]:
            tmp = staging_dir / (tmp_name + '.tmp')
            tmp.write_text(_json.dumps(contenido, indent=2, ensure_ascii=False, allow_nan=False))
            tmp.replace(staging_dir / tmp_name)

        # warnings_detail.jsonl
        wpath = staging_dir / 'warnings_detail.jsonl'
        tmp = wpath.with_suffix('.jsonl.tmp')
        with tmp.open('w', encoding='utf-8') as f:
            for w in warnings_pipeline:
                f.write(_json.dumps(to_jsonable(w), ensure_ascii=False, allow_nan=False) + '\n')
        tmp.replace(wpath)

        # ---------- 16. Promocion staging -> final_dir ----------
        # Backup de final previo si existe y es valido
        if final_dir.exists():
            backup_dir = final_root / f'_backup_v0_5_{run_id}' / nombre
            backup_dir.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(final_dir), str(backup_dir))
            log_fn(f'  backup previo: {backup_dir}')
        final_dir.parent.mkdir(parents=True, exist_ok=True)
        # Ajustar shard_paths del manifest tras la promocion (de staging a final)
        new_shard_paths = {}
        for sn, paths in shards_paths_por_split.items():
            new_shard_paths[sn] = [p.replace(str(staging_dir), str(final_dir)) for p in paths]
        manifest['shard_paths'] = new_shard_paths
        # Reescribir manifest en staging con paths corregidos antes de mover
        tmp = staging_dir / 'manifest.json.tmp'
        tmp.write_text(_json.dumps(to_jsonable(manifest), indent=2,
                                    ensure_ascii=False, allow_nan=False))
        tmp.replace(staging_dir / 'manifest.json')
        # Mover staging a final
        shutil.move(str(staging_dir), str(final_dir))

        # ---------- 17. done.flag al ULTIMO ----------
        (final_dir / 'done.flag').write_text(f'{PIPELINE_VERSION}\n{run_id}\n')

        log_fn(f'  PASS {nombre}: windows={n_windows_total}, shards={actual_n_shards}, '
               f'size={actual_size_mb_total} MB, roundtrip={roundtrip_info["result"]}')

        # Fila summary
        fila_summary = {
            'dataset':              nombre,
            'role':                 role,
            'client':               client_name,
            'dominio':              dominio,
            'evaluation_tier':      evaluation_tier,
            'status':               'PASS',
            'audit_version':        audit_version,
            'pipeline_version':     PIPELINE_VERSION,
            'pipeline_code_version': pipeline_code_version,
            'pipeline_config_hash': pipeline_config_hash,
            'run_mode':             run_mode,
            'tail_policy':          TAIL_POLICY,
            'n_channels':           n_canales,
            'n_units_total':        sum(n_units_por_split.values()),
            'n_windows_total':      n_windows_total,
            'n_temporal_patches':   n_temporal_patches,
            'n_channel_patches':    n_channel_patches_valid,
            'n_dense_temporal_patches': pilot_dense_temporal_patches,
            'n_dense_channel_patches':  pilot_dense_channel_patches,
            'invalid_patch_ratio':  invalid_patch_ratio,
            'dense_vs_valid_ratio': dense_vs_valid_ratio,
            'padding_ratio':        round(padding_ratio_actual, 4),
            'source_rows_total':    source_rows_total,
            'source_rows_covered_unique': source_rows_covered_unique,
            'dropped_tail_rows':    dropped_tail_rows,
            'estimated_n_shards':   audit_estimated_n_shards,
            'actual_n_shards':      actual_n_shards,
            'actual_size_mb_total': actual_size_mb_total,
            'warnings_count':       len(warnings_pipeline),
            'anti_leakage_status':  'pass',
            'roundtrip_status':     roundtrip_info.get('result', 'unknown'),
            'manifest_path':        str(final_dir / 'manifest.json'),
            'done_flag_path':       str(final_dir / 'done.flag'),
        }
        return 'PASS', fila_summary, {}

    finally:
        try:
            limpiar_cache_dataset(nombre, ficheros)
        except Exception:
            pass
        # Si quedo staging (por error antes de promocionar), no lo borramos
        # automaticamente: deja rastro para diagnostico.


print('procesar_dataset v0.5 definida (streaming + staging + roundtrip + asserts duros).')


## 10. Bucle: plan (`dry_run`) o procesamiento (`smoke`/`full`)

`RUN_MODE="dry_run"` produce `results/harmonization_full_plan.csv` con accion prevista por dataset (PROCESS / SKIP / FORCE / STALE) y motivo, **sin cargar PHMD ni escribir shards**.

`RUN_MODE="smoke"`/`"full"` itera `datasets_a_procesar` y llama a `procesar_dataset` con run_id comun para que todos los staging compartan el mismo identificador.


In [ ]:
import time
import traceback

def log(msg):
    log_path = LOG_DIR / f'{RUN_MODE}_{RUN_ID}.log'
    linea = f'[{datetime.now(timezone.utc).isoformat(timespec="seconds")}] {msg}'
    print(linea)
    with log_path.open('a') as f:
        f.write(linea + '\n')


def _accion_prevista(nombre, final_dir):
    """Decide action sin cargar PHMD: PROCESS / SKIP / FORCE / STALE."""
    if FORCE_REPROCESS_ALL or nombre in FORCE_REPROCESS_DATASETS:
        return 'FORCE', 'forzado por usuario'
    if not final_dir.exists():
        return 'PROCESS', 'no existe final_dir'
    manifest_p = final_dir / 'manifest.json'
    done_p     = final_dir / 'done.flag'
    if not manifest_p.exists():
        return 'STALE', 'no existe manifest.json'
    if not done_p.exists():
        return 'STALE', 'no existe done.flag'
    try:
        m = _json.loads(manifest_p.read_text())
    except Exception as e:
        return 'STALE', f'manifest no se puede leer: {e}'
    if m.get('audit_version') != AUDIT_VERSION_ESPERADO:
        return 'STALE', f'audit_version={m.get("audit_version")} != {AUDIT_VERSION_ESPERADO}'
    if m.get('pipeline_version') != PIPELINE_VERSION:
        return 'STALE', f'pipeline_version={m.get("pipeline_version")} != {PIPELINE_VERSION}'
    if m.get('tail_policy') != TAIL_POLICY:
        return 'STALE', f'tail_policy={m.get("tail_policy")} != {TAIL_POLICY}'
    if m.get('done') is True:
        return 'STALE', 'manifest tiene done=True (no permitido v0.5)'
    return 'SKIP', 'manifest + done.flag presentes con misma config basica'


# ---------- DRY RUN: solo plan ----------
if RUN_MODE == 'dry_run':
    log(f'DRY RUN: generando plan para {len(datasets_a_procesar)} datasets.')
    plan_rows = []
    for nombre in datasets_a_procesar:
        fila_audit = df_audit[df_audit['dataset'] == nombre].iloc[0]
        role = fila_audit['role']
        dominio = fila_audit.get('dominio') if pd.notna(fila_audit.get('dominio')) else None
        tier = fila_audit.get('evaluation_tier') if pd.notna(fila_audit.get('evaluation_tier')) else None
        client = dataset_to_client.get(nombre) if role == 'PRETRAIN_SOURCE' else None
        # Decidir output_dir
        if RUN_MODE == 'smoke':
            final_dir = PROCESSED_ROOT_SMOKE / nombre
        else:
            final_dir = PROCESSED_ROOT_CANONICAL / nombre
        accion, motivo = _accion_prevista(nombre, final_dir)
        plan_rows.append({
            'dataset':              nombre,
            'role':                 role,
            'dominio':              dominio,
            'evaluation_tier':      tier,
            'client':               client,
            'n_filas_total':        to_jsonable(fila_audit['n_filas_total']),
            'n_canales':            to_jsonable(fila_audit['n_canales']),
            'n_windows':            to_jsonable(fila_audit['n_windows']),
            'padding_ratio':        to_jsonable(fila_audit['padding_ratio']),
            'n_channel_patches':    to_jsonable(fila_audit['n_channel_patches']),
            'n_dense_channel_patches': to_jsonable(fila_audit['n_dense_channel_patches']),
            'estimated_size_mb':    to_jsonable(fila_audit.get('estimated_size_mb')),
            'estimated_n_shards':   to_jsonable(fila_audit['estimated_n_shards']),
            'action':               accion,
            'output_dir':           str(final_dir),
            'reason':               motivo,
        })
    df_plan = pd.DataFrame(plan_rows)
    # Orden recomendado: por estimated_size_mb ascendente (menos pesados primero)
    df_plan = df_plan.sort_values('estimated_size_mb', ascending=True, na_position='last').reset_index(drop=True)
    PLAN_CSV.parent.mkdir(parents=True, exist_ok=True)
    tmp = PLAN_CSV.with_suffix('.csv.tmp')
    df_plan.to_csv(tmp, index=False)
    tmp.replace(PLAN_CSV)
    log(f'plan guardado: {PLAN_CSV} ({len(df_plan)} filas)')
    print()
    print(df_plan.to_string(index=False))
    print()
    print(f'Plan generado para {RUN_MODE}. No se ha cargado PHMD ni escrito shards.')

else:
    log(f'Inicio {RUN_MODE} v{PIPELINE_VERSION} sobre {len(datasets_a_procesar)} datasets. RUN_ID={RUN_ID}')
    filas_summary = []
    fallos = []
    for i, nombre in enumerate(datasets_a_procesar, 1):
        log(f'[{i}/{len(datasets_a_procesar)}] {nombre}')
        t0 = time.time()
        fila_audit_row = df_audit[df_audit['dataset'] == nombre]
        if len(fila_audit_row) != 1:
            log(f'  AVISO: {nombre} no esta en audit (o duplicado). Saltando.')
            continue
        fila_audit = fila_audit_row.iloc[0]
        force = (FORCE_REPROCESS_ALL or nombre in FORCE_REPROCESS_DATASETS)
        try:
            status, fila, info = procesar_dataset(
                nombre, fila_audit, log,
                run_mode=RUN_MODE, run_id=RUN_ID,
                write_canonical=WRITE_CANONICAL_OUTPUTS,
                processed_root_canonical=PROCESSED_ROOT_CANONICAL,
                processed_root_smoke=PROCESSED_ROOT_SMOKE,
                dataset_to_client=dataset_to_client,
                force_reprocess=force,
            )
            elapsed = round(time.time() - t0, 1)
            log(f'  -> {status} en {elapsed}s')
            if fila is not None:
                filas_summary.append(fila)
        except Exception as e:
            elapsed = round(time.time() - t0, 1)
            tb = traceback.format_exc()
            fallos.append({'dataset': nombre, 'error': f'{type(e).__name__}: {e}',
                           'traceback': tb[-2500:], 'elapsed_s': elapsed})
            log(f'  FALLO {nombre}: {type(e).__name__}: {e}')
            if FAIL_FAST:
                log('FAIL_FAST=True. Abortando bucle.')
                break
    log(f'Bucle terminado. ok={len(filas_summary)}, fallos={len(fallos)}.')


## 11. Agregacion `processed_summary` + `client_summary` + asserts globales

En `dry_run` no se agrega nada (solo se genera el plan).

En `smoke`:
- escribe `processed_summary_smoke.csv` y `client_summary_smoke.csv`
- no toca artefactos canonicos
- `client_summary_smoke` contiene solo los clientes con al menos un PS procesado

En `full`:
- escribe `processed_summary.csv` y `client_summary.csv` canonicos
- **asserts globales obligatorios** antes de marcar el full como cerrado:
  - 47 filas (36 PS + 11 TT)
  - todos `status in {PASS, SKIP_VALID}`, ningun FAIL
  - 10 clientes en `client_summary`
  - sumas PS coinciden con `audit_summary` (temporal, channel, dense)
  - ningun fallo registrado


In [ ]:
def _output_paths_summary(run_mode):
    if run_mode == 'smoke':
        return PROCESSED_SUMMARY_SMOKE, CLIENT_SUMMARY_SMOKE
    return PROCESSED_SUMMARY_CANON, CLIENT_SUMMARY_CANON


def _escribir_csv_atomico(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.csv.tmp')
    df.to_csv(tmp, index=False)
    tmp.replace(path)


if RUN_MODE == 'dry_run':
    print(f'DRY RUN: plan en {PLAN_CSV}. No se agregan summaries.')

else:
    processed_summary_path, client_summary_path = _output_paths_summary(RUN_MODE)

    if not filas_summary:
        log('AVISO: sin filas en summary. No se escribe processed_summary*.')
    else:
        df_summary = pd.DataFrame(filas_summary).sort_values('dataset').reset_index(drop=True)
        _escribir_csv_atomico(df_summary, processed_summary_path)
        log(f'processed_summary{"_smoke" if RUN_MODE=="smoke" else ""}.csv guardado: '
            f'{processed_summary_path} ({len(df_summary)} filas)')
        print(df_summary[['dataset', 'role', 'client', 'status', 'n_windows_total',
                          'n_channel_patches', 'actual_n_shards', 'roundtrip_status']])

    # client_summary
    if filas_summary:
        df_summary_ps = pd.DataFrame([f for f in filas_summary if f.get('role') == 'PRETRAIN_SOURCE'])
        if not df_summary_ps.empty:
            # Agregamos solo PS realmente PASS o SKIP_VALID, sin fallback al audit
            agg = df_summary_ps.groupby('client', dropna=False).agg(
                n_datasets=('dataset', 'count'),
                n_units=('n_units_total', 'sum'),
                n_windows_total=('n_windows_total', 'sum'),
                n_temporal_patches=('n_temporal_patches', 'sum'),
                n_channel_patches=('n_channel_patches', 'sum'),
                n_dense_temporal_patches=('n_dense_temporal_patches', 'sum'),
                n_dense_channel_patches=('n_dense_channel_patches', 'sum'),
                actual_size_mb_total=('actual_size_mb_total', 'sum'),
            ).reset_index()
            total_temp = agg['n_temporal_patches'].sum() or 1
            total_chan = agg['n_channel_patches'].sum() or 1
            agg['w_temp_pct'] = (100.0 * agg['n_temporal_patches'] / total_temp).round(2)
            agg['w_chan_pct'] = (100.0 * agg['n_channel_patches'] / total_chan).round(2)
            agg = agg.sort_values('w_chan_pct', ascending=False).reset_index(drop=True)
            _escribir_csv_atomico(agg, client_summary_path)
            log(f'client_summary{"_smoke" if RUN_MODE=="smoke" else ""}.csv guardado: '
                f'{client_summary_path} ({len(agg)} clientes PS)')
            print()
            print(agg)
        else:
            log('AVISO: no hay PRETRAIN_SOURCE en filas_summary. No se escribe client_summary.')

    # ---------- ASSERTS GLOBALES (solo full) ----------
    if RUN_MODE == 'full':
        log('--- Asserts globales del full v0.5 ---')
        assert not fallos, f'FAIL_GLOBAL: hay {len(fallos)} datasets con error'
        assert len(filas_summary) == 47, f'processed_summary tiene {len(filas_summary)} filas, esperado 47'
        ps_count = sum(1 for f in filas_summary if f.get('role') == 'PRETRAIN_SOURCE')
        tt_count = sum(1 for f in filas_summary if f.get('role') == 'TRANSFER_TARGET')
        assert ps_count == 36 and tt_count == 11, f'PS/TT: {ps_count}/{tt_count}'
        bad_status = [f['dataset'] for f in filas_summary if f.get('status') not in ('PASS', 'SKIP_VALID')]
        assert not bad_status, f'datasets con status invalido: {bad_status}'

        # 10 clientes
        df_summary_ps = pd.DataFrame([f for f in filas_summary if f.get('role') == 'PRETRAIN_SOURCE'])
        unique_clients = sorted(df_summary_ps['client'].dropna().unique().tolist())
        assert len(unique_clients) == 10, f'clientes: {len(unique_clients)} != 10 ({unique_clients})'

        # Sumas PS == audit_summary
        sum_temp = int(df_summary_ps['n_temporal_patches'].sum())
        sum_chan = int(df_summary_ps['n_channel_patches'].sum())
        sum_d_temp = int(df_summary_ps['n_dense_temporal_patches'].sum())
        sum_d_chan = int(df_summary_ps['n_dense_channel_patches'].sum())
        assert sum_temp == AUDIT_TOTALS_PS['total_temporal_patches_ps'], (
            f'sum_temp_ps={sum_temp} vs audit={AUDIT_TOTALS_PS["total_temporal_patches_ps"]}')
        assert sum_chan == AUDIT_TOTALS_PS['total_channel_patches_ps'], (
            f'sum_chan_ps={sum_chan} vs audit={AUDIT_TOTALS_PS["total_channel_patches_ps"]}')
        assert sum_d_temp == AUDIT_TOTALS_PS['total_dense_temporal_patches_ps'], (
            f'sum_dense_temp={sum_d_temp} vs audit={AUDIT_TOTALS_PS["total_dense_temporal_patches_ps"]}')
        assert sum_d_chan == AUDIT_TOTALS_PS['total_dense_channel_patches_ps'], (
            f'sum_dense_chan={sum_d_chan} vs audit={AUDIT_TOTALS_PS["total_dense_channel_patches_ps"]}')
        log('--- Asserts globales PASS. Full cerrado. ---')

    elif RUN_MODE == 'smoke':
        log('--- Asserts smoke ---')
        if fallos:
            raise RuntimeError(f'SMOKE FAIL: {len(fallos)} datasets con error: '
                                f'{[f["dataset"] for f in fallos]}')
        bad_status = [f['dataset'] for f in filas_summary if f.get('status') not in ('PASS', 'SKIP_VALID')]
        assert not bad_status, f'smoke: status invalido en {bad_status}'
        log(f'--- Smoke PASS: {len(filas_summary)} datasets, sin fallos. ---')


## 12. Cierre

Resumen del run + verificacion de no contaminacion de outputs canonicos en `smoke`.


In [ ]:
print(f'=== Full v{PIPELINE_VERSION} | RUN_MODE={RUN_MODE} | RUN_ID={RUN_ID} ===')
print(f'TAIL_POLICY        = {TAIL_POLICY}')
print(f'audit_version      = {AUDIT_VERSION_ESPERADO}')
print(f'Datasets en plan   : {len(datasets_a_procesar)}')
if RUN_MODE != "dry_run":
    print(f'Procesados con exito: {len(filas_summary)}')
    print(f'Fallos               : {len(fallos)}')
print()
print('Outputs en Drive:')
if RUN_MODE == 'smoke':
    print(f'  {PROCESSED_ROOT_SMOKE}/<DATASET>/manifest.json, anti_leakage_report.json, *.tar, done.flag')
elif RUN_MODE == 'full':
    print(f'  {PROCESSED_ROOT_CANONICAL}/<DATASET>/manifest.json, anti_leakage_report.json, *.tar, done.flag')
print()
print('Outputs en repo:')
if RUN_MODE == 'dry_run':
    print(f'  {PLAN_CSV}')
elif RUN_MODE == 'smoke':
    print(f'  {PROCESSED_SUMMARY_SMOKE}')
    print(f'  {CLIENT_SUMMARY_SMOKE}')
    # Sanity check: no haber tocado canonicos
    if PROCESSED_SUMMARY_CANON.exists():
        canon_mtime = PROCESSED_SUMMARY_CANON.stat().st_mtime
        print(f'  (canonical {PROCESSED_SUMMARY_CANON.name} ultimo mtime={datetime.fromtimestamp(canon_mtime).isoformat()})')
    print(f'  CHECK: smoke NO debe haber escrito en {PROCESSED_SUMMARY_CANON} ni en {PROCESSED_ROOT_CANONICAL}.')
elif RUN_MODE == 'full':
    print(f'  {PROCESSED_SUMMARY_CANON}')
    print(f'  {CLIENT_SUMMARY_CANON}')
print()
print('Politica: este notebook NO ejecuta commit ni push.')

if RUN_MODE == 'dry_run':
    print()
    print('Siguiente paso recomendado: RUN_MODE="smoke" para validar en datos reales.')
elif RUN_MODE == 'smoke':
    print()
    print('Si smoke PASS:')
    print('  1. Verificar que ningun fichero canonical fue tocado (mtime).')
    print('  2. Verificar que processed_summary_smoke.csv tiene N filas con status PASS.')
    print('  3. Inspeccionar shards en {PROCESSED_ROOT_SMOKE}/CMAPSS/ y validar manualmente uno.')
    print('  4. Cambiar RUN_MODE="full" y reejecutar.')
elif RUN_MODE == 'full':
    print()
    print('Full cerrado. Commit + push manual de:')
    print(f'  - {PROCESSED_SUMMARY_CANON}')
    print(f'  - {CLIENT_SUMMARY_CANON}')
